# M3 · ① SIF 고위험요인 기초 EDA

먼저 `M2_전처리/02_M2_SIF_전처리.ipynb`를 실행하세요. 이 노트북은 실제 SIF 사례의 업종·기인물·고위험작업·재해유발요인을 살핍니다. **사례 수는 위험률이 아닙니다.** 노출 인원·작업량 분모가 없으므로 업종 간 위험 확률을 단정하지 않습니다.

산출물은 개인 `figures/`의 HTML 그래프와 `reports/`의 집계 보고서입니다.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'VS Code에서 SANUP-P 폴더를 열고 커널을 .venv로 선택하세요.'
PERSONAL = ROOT / 'data' / 'personal'
RAW = PERSONAL / 'raw' / 'source_01' / 'source_01_sif_archive.xlsx'
PROCESSED = PERSONAL / 'processed' / 'source_01_sif_cases.parquet'
assert RAW.exists(), f'① SIF 원본이 없습니다: {RAW}'
PROCESSED.parent.mkdir(parents=True, exist_ok=True)
print('원본:', RAW.name, f'{RAW.stat().st_size:,} bytes')

assert PROCESSED.exists(), f'M2 전처리 노트북을 먼저 실행하세요: {PROCESSED}'
cases = pd.read_parquet(PROCESSED)
required = {'record_id', 'sheet_group', 'industry_major', 'incident_summary', 'causal_object',
            'hazard_work', 'work_name', 'accident_type', 'trigger_factor'}
assert required <= set(cases.columns), required - set(cases.columns)
print('사례 수:', len(cases), '고유 ID:', cases['record_id'].nunique())

## 1. 사례 범위·결측·기술 통계

제조업 등·건설업의 열 구성이 다릅니다. 전체 결측률은 원본 시트에 없는 열을 포함하므로 시트별로 읽습니다.

In [ ]:
display(cases.groupby('sheet_group').size().rename('사례 수').to_frame())
display(cases.groupby(['sheet_group', 'industry_major'], dropna=False).size().rename('사례 수').sort_values(ascending=False).head(20).to_frame())
for group, frame in cases.groupby('sheet_group'):
    print(group, '사례:', len(frame))
    display((frame.isna().mean().mul(100).round(1).rename('결측률 %')).to_frame().T)

## 2. 기인물·작업·재해종류 분포

각 표는 **이 아카이브에 수록된 사례의 빈도**입니다. 인구·근로자·사업장 분모가 없어 발생률로 해석하지 않습니다.

In [ ]:
import plotly.express as px

figures = PERSONAL / 'figures'
figures.mkdir(parents=True, exist_ok=True)

def top_table(frame, column, n=12):
    return frame[column].dropna().value_counts().head(n).rename_axis(column).reset_index(name='사례 수')

objects = top_table(cases, 'causal_object')
manufacturing = cases[cases['sheet_group'] == '제조업 등']
construction = cases[cases['sheet_group'] == '건설업']
hazards = top_table(manufacturing, 'hazard_work')
works = top_table(construction, 'work_name')
accidents = top_table(construction, 'accident_type')
for title, table in [('기인물', objects), ('제조업 등 고위험작업', hazards),
                     ('건설업 작업명', works), ('건설업 재해종류', accidents)]:
    print(title)
    display(table)
    if not table.empty:
        fig = px.bar(table, x='사례 수', y=table.columns[0], orientation='h', title=title)
        fig.update_layout(yaxis={'categoryorder': 'total ascending'})
        fig.write_html(figures / f'source_01_{table.columns[0]}.html', include_plotlyjs='directory')
        display(fig)

## 3. 교차분석·사고 연도 확인

건설업의 재해종류와 기인물 교차표로 어떤 사례 조합이 자주 수록됐는지 봅니다. 사고 연도는 `재해개요` 문장의 `20XX년`에서만 추출하므로 추출 실패 건수를 같이 적습니다.

In [ ]:
top_objects = construction['causal_object'].value_counts().head(8).index
cross = pd.crosstab(
    construction.loc[construction['causal_object'].isin(top_objects), 'causal_object'],
    construction.loc[construction['causal_object'].isin(top_objects), 'accident_type'],
)
display(cross)
# 원문 재해개요의 앞부분에서 연도를 찾는다. 원본 문구는 고치지 않는다.
date_parts = cases['incident_summary'].fillna('').astype(str).str.extract(
    r'^\s*(?P<year>20\d{2})(?P<marker>년|월|[./-])'
)
years = date_parts['year']
year_counts = years.value_counts().sort_index()
malformed = date_parts['marker'].eq('월')
normal_year_marker = int(date_parts['marker'].eq('년').sum())
dot_date_marker = int(date_parts['marker'].eq('.').sum())
catalog_year_counts = years[years.between('2016', '2023') & ~malformed].value_counts().sort_index()
outside_catalog_window = years.notna() & ~years.between('2016', '2023')
outside_year_counts = years[outside_catalog_window].value_counts().sort_index()
print('연도 추출:', int(years.notna().sum()), '/', len(cases), '| 연도 표기 확인 필요:', int(malformed.sum()), '| 안내 기간(2016~2023) 밖:', int(outside_catalog_window.sum()))
display(catalog_year_counts.rename('안내 기간 내 사례 수').to_frame())
display(outside_year_counts.rename('안내 기간 밖 사례 수(별도 확인)').to_frame())
print('안내 기간 밖 사례 ID')
display(cases.loc[outside_catalog_window, ['record_id', 'sheet_group']].assign(재해개요_연도=years[outside_catalog_window].values))
print('연도 표기 확인 사례 ID')
display(cases.loc[malformed, ['record_id', 'sheet_group']].assign(원문_연도표기=date_parts.loc[malformed, 'marker'].values))


## 4. 집계·한계 기록

이 결과만으로 사고 위험의 인과관계나 업종별 실제 사망률을 계산하지 않습니다. 대시보드 ②~⑤와 ①은 기준기간·단위가 다릅니다.

In [ ]:
report = PERSONAL / 'reports' / 'source_01_sif_eda.md'
report.parent.mkdir(parents=True, exist_ok=True)
text = [
    '# ① SIF 기초 EDA', '',
    f'- 원본 사례: {len(cases):,}건 (제조업 등 {len(manufacturing):,}건, 건설업 {len(construction):,}건)',
    f'- 고유 record_id: {cases["record_id"].nunique():,}개',
    f'- 재해개요 첫머리에서 연도 추출: {int(years.notna().sum()):,}/{len(cases):,}건',
    f'- 안내 기간 내 정상 표기 연도별 집계: {int(catalog_year_counts.sum()):,}건; 기간 밖: {int(outside_catalog_window.sum()):,}건',
    f'- 연도 구분자: {normal_year_marker:,}건 년, {dot_date_marker:,}건 마침표(집계 포함), {int(malformed.sum()):,}건 월(집계 제외)',
    f'- 연도 표기가 `월`로 시작해 연도별 집계에서 제외한 사례: {int(malformed.sum()):,}건', '',
    '## 안내 기간(2016~2023) 내 연도별 사례 수',
    *[f'- {year}: {count:,}건' for year, count in catalog_year_counts.items()], '',
    '## 안내 기간 밖 연도별 사례 수',
    *[f'- {year}: {count:,}건' for year, count in outside_year_counts.items()],
    '- 기간 밖 54건은 2013~2015년 53건, 2024년 1건이다. 원문 날짜 표기를 보존하고 안내기간 분석에서는 별도로 분리했다.', '',
    '## 안내 기간 밖 사례 ID',
    *[f'- {rid} ({year})' for rid, year in zip(cases.loc[outside_catalog_window, 'record_id'], years[outside_catalog_window])], '',
    '## 연도 표기 확인 사례 ID',
    *[f'- {rid} ({summary[:100]})' for rid, summary in zip(cases.loc[malformed, 'record_id'], cases.loc[malformed, 'incident_summary'])], '',
    '## 해석 한계',
    '- 이 표는 SIF 아카이브 수록 사례 수이며 노출 분모가 없어 위험률을 뜻하지 않는다.',
    '- 재해개요 문장 앞부분의 연도는 사고 발생일을 모두 검증한 값이 아니다. 기간 분석은 잠정 결과다.',
    '- ②~⑤ 대시보드 자료와 기준연도·단위가 달라 직접 결합하지 않는다.',
    '- 원본 저작물의 출처표시·변경금지 조건을 적용해 개인 분석 범위에서만 다루고, 원본 파일은 수정하지 않는다.',
]
report.write_text('\n'.join(text) + '\n', encoding='utf-8')
print('보고서:', report)


## SIF 사례 분포 그래프

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from IPython.display import Image, display

ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'pyproject.toml').exists()),None)
assert ROOT is not None
if any(font.name=='Malgun Gothic' for font in font_manager.fontManager.ttflist):
    plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False
figures=ROOT/'data'/'personal'/'figures'; figures.mkdir(parents=True,exist_ok=True)
sif_plot=pd.read_parquet(ROOT/'data'/'personal'/'processed'/'source_01_sif_cases.parquet')

top_objects=sif_plot['causal_object'].value_counts().head(12).sort_values()
fig,ax=plt.subplots(figsize=(10,7))
ax.barh(top_objects.index.astype(str),top_objects.values,color='#1f6e8c')
ax.set(xlabel='아카이브 사례 수',title='① SIF · 기인물 상위 12개')
ax.bar_label(ax.containers[0],padding=3)
fig.tight_layout()
path=figures/'sif_causal_objects_top12.png'
fig.savefig(path,dpi=170,bbox_inches='tight'); plt.close(fig)
display(Image(filename=str(path)))

accidents=sif_plot['accident_type'].dropna().value_counts().head(10).sort_values()
fig,ax=plt.subplots(figsize=(10,6))
ax.barh(accidents.index.astype(str),accidents.values,color='#b65f38')
ax.set(xlabel='건설업 시트 사례 수',title='① SIF · 건설업 사고 유형 상위 10개')
ax.bar_label(ax.containers[0],padding=3)
fig.tight_layout()
path=figures/'sif_construction_accident_types.png'
fig.savefig(path,dpi=170,bbox_inches='tight'); plt.close(fig)
display(Image(filename=str(path)))

## 분석 결과와 해석 범위

- 원본에서 제조업 등 2,573건, 건설업 3,459건을 확인했다. 합계 6,032건이고 record_id 중복은 없었다. 이 구성은 아카이브 수록 비중이지 업종별 실제 사고율은 아니다.
- 기인물 최다 항목은 `분류 불가` 398건(6.6%)이다. 분류된 기인물 중 지게차 216건, 비계 196건, 굴착기(백호우) 186건이 많았다. 분류 불가 항목도 결과표에 남겨 데이터의 한계를 같이 보이게 했다.
- 건설업 시트는 추락이 2,030건(58.7%)으로 가장 많았다. 사례 내 비중이므로 작업량·근로자 수를 반영한 위험도로 말할 수 없다. 기인물×재해종류 교차표도 사례 조합의 빈도 확인용이다.
- 원문 첫머리의 연도 구분자는 6,026건이 ‘년’, 1건이 마침표 날짜 형식, 5건이 잘못된 ‘월’ 표기였다. 마침표 형식의 SIF-C-2800은 2020.10.24로 정상 처리했다. 잘못된 ‘월’ 표기 5건은 연도별 집계에서 제외하고 별도 확인 대상으로 남겼다.
- 안내기간 밖 54건은 원문에 적힌 사건 연도가 2013~2015년 53건과 2024년 1건이다. 전체 사례와 원문은 보존하고, 공식 안내기간(2016~2023) 연도별 표에서는 기간 밖 54건과 잘못 표기된 5건을 제외했다. 안내기간에 포함된 정상 표기 사례 수는 5,973건이다.
- 그래프는 개인 작업물의 `data/personal/figures/`에 저장한다. 원본 XLSX는 변경하지 않고, 시각화·분석 산출물도 출처표시 및 변경금지 조건을 지켜 개인 범위에서 사용한다.
- ②~⑤ 집계자료와 ① 사례자료는 기준 기간과 관측 단위가 다르다. 업종 이름이 비슷하다는 이유로 사례 수와 사망만인율을 같은 분모처럼 합치지 않는다.

### 다음에 쓸 때 확인할 것

- 연도 분석을 발표 자료에 넣기 전 공식 원본의 버전·기간·날짜 표기를 확인한다.
- SIF 빈도는 “이 파일에 몇 건 실렸는지”로 설명하고 사고 확률·위험률로 표현하지 않는다.
- 398건의 `분류 불가`와 원문 날짜 표기 문제를 숨기지 않는다. 날짜를 정정해야 한다면 별도 정제 열과 근거를 남기고 원문 열은 보존한다.
- M3 통계는 `04_M3_통계_관계분석.ipynb`, 문장 키워드 탐색은 `05_M3_SIF_TFIDF.ipynb`, 대시보드용 정적 그래프는 `06_대시보드_v1_설계.ipynb`에서 이어간다.
